In [0]:

dbutils.widgets.removeAll()

In [0]:
# Creacion de widgets para creacion de variables
# PARAMETRIZAR ADLS Y CATALOGO A PROD
dbutils.widgets.text("PRM_nameStorage","adlsecompag")
dbutils.widgets.text("PRM_catalogo","catalogo_desa_intEcommerce")

In [0]:

PRM_nameStorage = dbutils.widgets.get("PRM_nameStorage")
PRM_catalogo = dbutils.widgets.get("PRM_catalogo")

In [0]:

spark.sql(f"use catalog {PRM_catalogo}")

In [0]:

tablas_a_truncar = [
    "bronze_v.Clientes_Sistema",
    "bronze_v.Interaccion_Sistema",
    "bronze_v.Productos_Sistema",
    "bronze_v.ecommerce_data",
    "silver_v.Tabla_Cliente",
    "silver_v.Tabla_Destipinteraccion",
    "silver_v.Tabla_Producto",
    "silver_v.Tabla_IntEcommerce",
    "golden_v.Interaccion_Analisis",
    "golden_v.categoria_top_ecommerce",
    "golden_v.clientes_top_compras",
    "bronze_vdc.Clientes_Sistema",
    "silver_vdc.Tabla_Cliente",
    "golden_vdc.Interaccion_Analisis",
    "golden_vdc.clientes_top_compras"
]

for tabla in tablas_a_truncar:
    full_table_path = f"{PRM_catalogo}.{tabla}"
    
    try:
        if spark.catalog.tableExists(full_table_path):
            spark.sql(f"TRUNCATE TABLE {full_table_path}")
            print(f"Tabla existente (Se truncará los datos): {full_table_path}")
        else:
            print(f"Tabla no existe: {full_table_path}")
    except Exception as e:
        print(f"Error en {full_table_path}: {e}")

In [0]:

spark.sql(f"""CREATE OR REPLACE TABLE {PRM_catalogo}.bronze_v.Clientes_Sistema	
USING DELTA
LOCATION 'abfss://cont-ecom@{PRM_nameStorage}.dfs.core.windows.net/bronze_v/tablas/Clientes_Sistema'
AS SELECT 
User_id as User_id,
sha2(Last_Name,256) as Last_Name,
sha2(Name,256) as Name,
sha2(CAST(Age AS STRING), 256) as Age,
sha2(Cell_number,256) as Cell_number,
Sign_date as Sign_date,
Fecha_proceso as Fecha_proceso
FROM {PRM_catalogo}.bronze.Clientes_Sistema;
""")

spark.sql(f"""CREATE OR REPLACE TABLE {PRM_catalogo}.bronze_v.Interaccion_Sistema		
USING DELTA
LOCATION 'abfss://cont-ecom@{PRM_nameStorage}.dfs.core.windows.net/bronze_v/tablas/Interaccion_Sistema'
AS SELECT * FROM {PRM_catalogo}.bronze.Interaccion_Sistema;
""")

spark.sql(f"""CREATE OR REPLACE TABLE {PRM_catalogo}.bronze_v.Productos_Sistema			
USING DELTA
LOCATION 'abfss://cont-ecom@{PRM_nameStorage}.dfs.core.windows.net/bronze_v/tablas/Productos_Sistema'
AS SELECT * FROM {PRM_catalogo}.bronze.Productos_Sistema;
""")

spark.sql(f"""CREATE OR REPLACE TABLE {PRM_catalogo}.bronze_v.ecommerce_data			
USING DELTA
LOCATION 'abfss://cont-ecom@{PRM_nameStorage}.dfs.core.windows.net/bronze_v/tablas/ecommerce_data'
AS SELECT * FROM {PRM_catalogo}.bronze.ecommerce_data;
""")


In [0]:

spark.sql(f"""CREATE OR REPLACE TABLE {PRM_catalogo}.bronze_vdc.Clientes_Sistema	
USING DELTA
LOCATION 'abfss://cont-ecom@{PRM_nameStorage}.dfs.core.windows.net/bronze_vdc/tablas/Clientes_Sistema'
AS SELECT 
User_id as User_id,
Last_Name as Last_Name,
Name as Name,
Age as Age,
Cell_number as Cell_number,
Sign_date as Sign_date,
Fecha_proceso as Fecha_proceso
FROM {PRM_catalogo}.bronze.Clientes_Sistema;
""")


In [0]:

spark.sql(f"""CREATE OR REPLACE TABLE {PRM_catalogo}.silver_v.Tabla_Cliente	
USING DELTA
LOCATION 'abfss://cont-ecom@{PRM_nameStorage}.dfs.core.windows.net/silver_v/tablas/Tabla_Cliente'
AS SELECT 
ID_Cliente as ID_Cliente,
sha2(Apellido,256) as Apellido_hashed,
sha2(Nombre,256) as Nombre_hashed,
sha2(CAST(Edad AS STRING), 256) as Edad_hashed,
sha2(Numero_Celular,256) as Numero_Celular_hashed,
Fecha_inscripcion_PagWeb as Fecha_inscripcion_PagWeb,
Fecha_proceso as Fecha_proceso
FROM {PRM_catalogo}.silver.Tabla_Cliente;
""")

spark.sql(f"""CREATE OR REPLACE TABLE {PRM_catalogo}.silver_v.Tabla_Producto		
USING DELTA
LOCATION 'abfss://cont-ecom@{PRM_nameStorage}.dfs.core.windows.net/silver_v/tablas/Tabla_Producto'
AS SELECT * FROM {PRM_catalogo}.silver.Tabla_Producto;
""")

spark.sql(f"""CREATE OR REPLACE TABLE {PRM_catalogo}.silver_v.Tabla_Destipinteraccion			
USING DELTA
LOCATION 'abfss://cont-ecom@{PRM_nameStorage}.dfs.core.windows.net/silver_v/tablas/Tabla_Destipinteraccion'
AS SELECT * FROM {PRM_catalogo}.silver.Tabla_Destipinteraccion;
""")

spark.sql(f"""CREATE OR REPLACE TABLE {PRM_catalogo}.silver_v.Tabla_IntEcommerce			
USING DELTA
LOCATION 'abfss://cont-ecom@{PRM_nameStorage}.dfs.core.windows.net/silver_v/tablas/Tabla_IntEcommerce'
AS SELECT * FROM {PRM_catalogo}.silver.Tabla_IntEcommerce;
""")

In [0]:

spark.sql(f"""CREATE OR REPLACE TABLE {PRM_catalogo}.silver_vdc.Tabla_Cliente	
USING DELTA
LOCATION 'abfss://cont-ecom@{PRM_nameStorage}.dfs.core.windows.net/silver_vdc/tablas/Tabla_Cliente'
AS SELECT *
FROM {PRM_catalogo}.silver.Tabla_Cliente;
""")

In [0]:

spark.sql(f"""CREATE OR REPLACE TABLE {PRM_catalogo}.golden_v.Interaccion_Analisis		
USING DELTA
LOCATION 'abfss://cont-ecom@{PRM_nameStorage}.dfs.core.windows.net/golden_v/tablas/Interaccion_Analisis'
AS SELECT * EXCEPT(Nombre_completo, Numero_Celular),
sha2(Nombre_completo,256) as Nombre_completo_hashed,
sha2(Numero_Celular, 256) as Numero_Celular_hashed
FROM {PRM_catalogo}.golden.Interaccion_Analisis;
""")

spark.sql(f"""CREATE OR REPLACE TABLE {PRM_catalogo}.golden_v.Categoria_Top_Ecommerce			
USING DELTA
LOCATION 'abfss://cont-ecom@{PRM_nameStorage}.dfs.core.windows.net/golden_v/tablas/Categoria_Top_Ecommerce'
AS SELECT * FROM {PRM_catalogo}.golden.Categoria_Top_Ecommerce;
""")

spark.sql(f"""CREATE OR REPLACE TABLE {PRM_catalogo}.golden_v.Clientes_Top_Compras			
USING DELTA
LOCATION 'abfss://cont-ecom@{PRM_nameStorage}.dfs.core.windows.net/golden_v/tablas/Clientes_Top_Compras'
AS SELECT 
ID_Cliente as ID_Cliente,
sha2(Nombre_completo,256) as Nombre_completo_hashed,
sha2(Numero_Celular,256) as Numero_Celular_hashed,
KPI_Tipo_Cliente as KPI_Tipo_Cliente,
Precio_Compra_Total as Precio_Compra_Total
FROM {PRM_catalogo}.golden.Clientes_Top_Compras;
""")


In [0]:

spark.sql(f"""CREATE OR REPLACE TABLE {PRM_catalogo}.golden_vdc.Interaccion_Analisis	
USING DELTA
LOCATION 'abfss://cont-ecom@{PRM_nameStorage}.dfs.core.windows.net/golden_vdc/tablas/Interaccion_Analisis'
AS SELECT *
FROM {PRM_catalogo}.golden.Interaccion_Analisis;
""")

spark.sql(f"""CREATE OR REPLACE TABLE {PRM_catalogo}.golden_vdc.Clientes_Top_Compras	
USING DELTA
LOCATION 'abfss://cont-ecom@{PRM_nameStorage}.dfs.core.windows.net/golden_vdc/tablas/Clientes_Top_Compras'
AS SELECT *
FROM {PRM_catalogo}.golden.Clientes_Top_Compras;
""")